# 03 — LSTM pour la classification de sentiment
**Embedding entraînable + LSTM + dense**

Corrections par rapport au notebook original :
- API Keras moderne (plus de `model.predict_classes`, `history['acc']`, etc.)
- `train_test_split` avec `stratify` pour un split équilibré
- Validation split pendant l'entraînement
- Callback `EarlyStopping` + `ReduceLROnPlateau`
- Visualisations complètes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json, os

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, GlobalMaxPooling1D, SpatialDropout1D
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print(f'TensorFlow : {tf.__version__}')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
SEED = 42
tf.random.set_seed(SEED)

## 1. Chargement & preprocessing

In [ ]:
df = pd.read_csv(
    '../data/all-data.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'headline']
)
print(f'Shape : {df.shape}')
print(df['sentiment'].value_counts())
df.head()

In [ ]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean'] = df['headline'].apply(clean_text)
print('Exemple :', df['clean'].iloc[0])

## 2. Encodage des labels & split

In [ ]:
le = LabelEncoder()
le.fit(['positive', 'neutral', 'negative'])
df['label'] = le.transform(df['sentiment'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean'].values,
    df['label'].values,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label'].values
)

NUM_CLASSES = len(le.classes_)
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f'Train : {len(X_train_raw)} | Test : {len(X_test_raw)}')
print(f'Classes : {le.classes_}')

## 3. Tokenisation & padding

In [ ]:
MAX_VOCAB   = 15_000
MAX_SEQ_LEN = 50      # 95e percentile de la distribution de longueur
EMBED_DIM   = 64

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_raw),
    maxlen=MAX_SEQ_LEN, padding='post', truncating='post'
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test_raw),
    maxlen=MAX_SEQ_LEN, padding='post', truncating='post'
)

VOCAB_SIZE = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f'Vocabulaire utilisé : {VOCAB_SIZE}')
print(f'Shape X_train : {X_train.shape}')
print(f'Shape X_test  : {X_test.shape}')

## 4. Architecture LSTM bidirectionnel

In [ ]:
def build_model(vocab_size, embed_dim, seq_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=seq_len, mask_zero=True),
        SpatialDropout1D(0.2),
        Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)),
        Bidirectional(LSTM(32, dropout=0.2, recurrent_dropout=0.1)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(VOCAB_SIZE, EMBED_DIM, MAX_SEQ_LEN, NUM_CLASSES)
model.summary()

## 5. Entraînement

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    )
]

history = model.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

## 6. Courbes d'apprentissage

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    epochs = range(1, len(history.history['accuracy']) + 1)

    # Accuracy
    axes[0].plot(epochs, history.history['accuracy'],     label='Train', lw=2)
    axes[0].plot(epochs, history.history['val_accuracy'], label='Val',   lw=2, linestyle='--')
    axes[0].set_title('Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    # Loss
    axes[1].plot(epochs, history.history['loss'],     label='Train', lw=2)
    axes[1].plot(epochs, history.history['val_loss'], label='Val',   lw=2, linestyle='--')
    axes[1].set_title('Loss', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.suptitle('Courbes d\'apprentissage LSTM', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history)

## 7. Évaluation

In [ ]:
y_proba     = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_proba, axis=1)
acc_lstm    = accuracy_score(y_test, y_pred)
class_names = le.classes_.tolist()

print(f'Accuracy LSTM : {acc_lstm:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
annot = np.array([[f'{v}\n({p:.0%})' for v, p in zip(r_v, r_p)]
                  for r_v, r_p in zip(cm, cm_pct)])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=annot, fmt='', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, ax=ax
)
ax.set_title(f'Matrice de confusion LSTM\nAcc = {acc_lstm:.3f}',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

## 8. Test sur phrases personnalisées

In [ ]:
def predict_sentiment(texts, tokenizer, model, le, max_len=MAX_SEQ_LEN):
    cleaned = [clean_text(t) for t in texts]
    seqs    = tokenizer.texts_to_sequences(cleaned)
    padded  = pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')
    proba   = model.predict(padded, verbose=0)
    preds   = np.argmax(proba, axis=1)
    for txt, pred, prob in zip(texts, preds, proba):
        label = le.inverse_transform([pred])[0]
        conf  = prob[pred]
        print(f'[{label.upper():8s}] ({conf:.2%})  →  {txt[:80]}')

test_sentences = [
    "The company reported record profits and strong revenue growth for Q3.",
    "Operating profit fell sharply amid rising costs and declining sales.",
    "The board announced no changes to the current dividend policy.",
    "Shares tumbled after the CEO resigned unexpectedly due to fraud allegations.",
    "The acquisition is expected to complete pending regulatory approval."
]
predict_sentiment(test_sentences, tokenizer, model, le)

In [ ]:
# Sauvegarde du score pour le notebook 04
os.makedirs('../results', exist_ok=True)
try:
    with open('../results/baseline_scores.json') as f:
        scores = json.load(f)
except FileNotFoundError:
    scores = {}

scores['LSTM'] = round(float(acc_lstm), 4)
with open('../results/baseline_scores.json', 'w') as f:
    json.dump(scores, f, indent=2)
print('Score LSTM sauvegardé :', scores)